- **Created:** [Rik Henson](https://www.mrc-cbu.cam.ac.uk/people/rik.henson/) with thanks to [Petar Raykov](https://www.mrc-cbu.cam.ac.uk/people/Petar.Raykov)
- **Date:** July 2026
- **conda environment**: This uses the [mri environment](https://github.com/RikHenson/PythonNeuroimagingCourse/blob/main/mri_environment.yml)

# State-based Functional Connectivity

This notebook introduces methods to measure functional connectivity based on fMRI timeseries recorded during various brain states, such as resting. This "time-series correlation" (TSC) between voxels or ROIs is measured across TRs, having removed confounds like motion artifacts, WM/CSF signal, global signal etc. Topics include: 1) multivariate decompositions (eg ICA), 2) seed-to-whole-brain mapping and 3) ROI-based connectomes.

Normally this type of analysis is done on resting-state data (or data from continuous stimuli like movie watching), but there are no such data in the Face Recognition fMRI dataset we are using. Nonetheless, we can simulate resting-state data by removing task effects (i.e., treating the task regressors in our design matrix as another confound). 

Note that there are other methods for task-based connectivity, like "Beta-Series Regression" (BSR) and "Psycho-Physiological Interactions" (PPIs), which are covered in a separate notebook. Note also that there are also more complex dynamical models (like DCM or HMMs) that get closer to measuring true "effective connectivity" (by simulating a full network of regions, see [Stephan & Friston (2011)](https://pmc.ncbi.nlm.nih.gov/articles/PMC3013343/) for example) - but these are not covered here. Finally, we are using unidimensional connectivity, averaging values across voxels within each ROI, though there are also multidimensional ways to estimate connectivity (see [Basti et al (2020)](https://pubmed.ncbi.nlm.nih.gov/32682988/) for example review).

## 0. Getting Ready

As usual, we need some python packages like below:

In [ ]:
import matplotlib.pyplot as plt # plotting
# to show plots in cell
%matplotlib inline   

import os           # To interact with the operating system, including files and paths (e.g. path.join)
import bids.layout  # To fetch data from BIDS-compliant datasets
import numpy as np  # This lets python process arrays/matrices
import pandas       # To use "dataframes"      
import nibabel      # Basic nifti image utilities

import nilearn                  # Many useful functions for MRI, including...
from nilearn import image       # to load (load_img), resample (resample_to_img), manipulate (math_img) fMRI data, etc.
from nilearn.maskers import NiftiMasker, NiftiMapsMasker # For extracting data from images
from nilearn import plotting    # includes plot_roi, plot_stat_map, view_img_on_surf, etc.
from nilearn import datasets    # for atlases below
from nilearn.connectome import ConnectivityMeasure
from nilearn.plotting import plot_prob_atlas

import scipy                     # statistical tools
from scipy.stats import pearsonr, zscore # like Pearson correlation and Z-scoring

from sklearn.decomposition import PCA, FastICA
from nilearn.decomposition import CanICA

import warnings 

And set-up our input and output directories:

In [ ]:
wd = '/home/cognestic/COGNESTIC/07_fMRI_Connectivity/data/' 
#wd = '/imaging/rh01/Methods/Cognestic/CognesticConnectivity' 
out_dir = os.path.join(wd, 'state_con')
if not os.path.exists(out_dir):
    os.makedirs(out_dir)
os.chdir(out_dir)
print(f"Working directory currently {os.getcwd()}")

fmri_data_dir = '/home/cognestic/COGNESTIC/06_fMRI/FaceRecognition/' 
#fmri_data_dir = '/imaging/correia/da05/workshops/2026-COGNESTIC/06_fMRI/FaceRecognition/'
fmri_data_dir = os.path.join(fmri_data_dir , 'data') # data in BIDS format
layout = bids.layout.BIDSLayout(fmri_data_dir, derivatives = True)

Now let's get the preprocessed Face Recognition fMRI data from one subject from previous notebooks:

In [ ]:
sID = '15' # same subject we used before
all_runs = layout.get(subject = sID, datatype = 'func', desc = 'preproc', extension = '.nii.gz', return_type = 'filename')
img = image.load_img(all_runs[0])
TR = layout.get_tr()
print("Found " + str(len(all_runs)) + " preprocessed functional files, each with " + str(img.shape[3]) + " volumes, sampled every " + str(TR) + "s")

## 1 Multivariate decomposition

Multivariate techniques like Principal Component Analysis (PCA) and Independent Component Analysis (ICA) can be used to estimate functional networks of voxels with similar timeseries (see [Smitha et al (2017)](https://pmc.ncbi.nlm.nih.gov/articles/PMC5524274) for example review), as well as possibly distinguish signal and noise components (in order to project out latter).

Nilearn does not have a specific module for ICA and just uses a the sklearn ICA module that can be applied to any numpy array data (see [Abraham et al. 2014](https://www.frontiersin.org/articles/10.3389/fninf.2014.00014/full#h7) that introduces nilearn). Note that fMRIprep can also run ICA AROMA to remove motion-related components (from FSL's Melodic toolbox). 

Once components are obtained, you can try to classify them as signal or noise (e.g., motion but also physiological noise). There are automated methods for this (such as in FSL's FIX), or you can do so manually, e.g., [Griffanti et al. 2017](https://doi.org/10.1016/j.neuroimage.2016.12.036).

Anyway, let's concatenate all 9 runs for this subject, and extract timeseries data using an EPI masker based on threshold the mean EPI (bold) images:

In [ ]:
epi_masker = NiftiMasker(mask_strategy="epi", smoothing_fwhm=8, 
                        standardize='zscore_sample', detrend=True) # Mean-correct and Z-score timeseries
concat_runs = image.concat_imgs(all_runs)

# Uncomment if you want to remove run effects (PCs will look better; see discussion below), but not done here for pedagogical purposes
# run_indices = np.repeat(np.arange(1, 10), 208)
# all_runs = image.clean_img(all_runs, runs=run_indices, detrend=True, standardize=None) # we will standardise across runs with masker
# note that you cannot use epi mask after detrending each run, so will have to redefine whole_brain_masker to use standard template
                        
timeseries = epi_masker.fit_transform(concat_runs)
print('Now %d timepoints for %d voxels' %(timeseries.shape[0], timeseries.shape[1]) )

Now we can start with PCA, which finds spatially-orthogonal modes:

In [ ]:
n_components = 20 # choosing the number of components is critical, rule of thumb is between 20-60 more often possibly between 20-30;
# FSL Melodic has an built-in option to estimate the number of components to keep, however this can over-estimate the number of components that are needed
# Choosing the number of components affects the functional 'nodes' or the networks extracted from ICA, as more components can potentially fragment a network into sub-networks
# read here for more information - https://neurostars.org/t/automatic-estimation-of-number-if-ica-components-in-nilearn-canica/287/6

pca = PCA(n_components=n_components, random_state=42)
# in sklearn and nilearn the shape is always n_samples x n_features
# to do spatial ICA, the direction considered as random is that of voxels and not the timepoints
# effectively we find statistically independent spatial maps and their associated time-courses
components_pca = pca.fit_transform(timeseries.T).T 
print('Now %d components for %d voxels' %(components_pca.shape[0], components_pca.shape[1]) )

We can plot the amount of variance explained by each component (related to the "eigen" or singular values of the singular-value decomposition underlying the PCA):

In [ ]:
plt.plot(pca.explained_variance_ratio_)
ax = plt.gca()
ax.set_xticks(range(0,20),["{:d}".format(x) for x in range(1,21)])
ax.set_title('Scree plot ratio')
print('%d components explain %d%% of the variance' %(n_components ,np.sum(pca.explained_variance_ratio_) * 100) )

We can also view the spatial (and temporal) modes associated with (some of) the components:

In [ ]:
components_pca_zscored = zscore(components_pca, axis=1) # Z-score for any plotting/thresholding
#components_pca_zscored[np.abs(components_pca_zscored) < 1] = 0 # here thresholding is arbitrary
pca_image = epi_masker.inverse_transform(components_pca_zscored)

#for comp in range(n_components): # If you want to look through all 20!
for comp in range(2): # Just first 2... (given hint of Scree change after 4, but mainly session effects (see below))
    fig = plt.figure(figsize=(20,10))
    display = plotting.plot_stat_map(image.index_img(pca_image, comp), figure = fig , threshold=0, cut_coords=[1,-23,-20], black_bg=0)

    display.title(f'PCA Component {comp+1}',size=30)
    display.annotate(size=30)
    cbar_ax = fig.axes[-1]
    cbar_ax.tick_params(labelsize=30)

Note that each spatial mode (map) above has a corresponding temporal mode (timecourse), for which the first two are:

In [ ]:
#for comp in range(n_components):
for comp in range(2):
    plt.plot(pca.components_[comp,:]);
    #plt.plot(pca.components_[comp,1:200]); # If want to zoom-in to see task blocks; 

You can see that these are dominated by "session" or "run" effects - i.e., large changes in signal every time scanner stopped and restarted. To avoid these, you could repeat the PCA, but first detrending the data within each run (ie uncommment "image.clean_img" line above).

Note also that the sign of the spatial and temporal components are inter-changeable, e.g, you get the same result by multiplying both the spatial and temporal mode of each component by -1 (so the red/blue colours in spatial maps are arbitrary).

Anyway, these maps are difficult to interpret. The first one for example could reflect motion artifacts, since there is high variance around the edge of the brain (e.g., owing to different head positions across runs, which will not be correctable by realignment if the magnetic field is inhomogeneous, eg nonlinear distortions of the images). Indeed, seeking spatially-orthogonal components may not be the best way to identify networks. So let's try ICA instead, which looks for spatially independent (rather than orthogonal) components (note this will take several minutes):

In [ ]:
ica = FastICA(n_components=n_components, random_state=42)
# in sklearn and nilearn the shape is always n_samples x n_feautres
# to do spatial ICA, the directon considered as random is that of voxels and not the timepoints
# effectively we find statistically independent spatial maps and their associated time-courses
components_ica = ica.fit_transform(timeseries.T).T # spatial ICA

Note that ICA actually starts with PCA to reduce dimensionality.

Now let's threshold the spatial maps (just for visualisation) and look at some selected components (you can uncomment code below to look at all):

In [ ]:
components_ica_zscored = zscore(components_ica,axis=1)
components_ica_zscored[np.abs(components_ica_zscored) < 1] = 0 # here thresholding is arbitrary, previously a strategy is to threshold the components to show the voxels that are Z > 2; However, this is not necessarily the most effective strategy
# since the spatial maps should be non-Gaussian by definition of being derived from ICA decomposition; FSL uses a Mixture of Gaussians to perform the thresholding see here https://www.fmrib.ox.ac.uk/datasets/techrep/tr02cb1/tr02cb1/node1.html
ica_image = epi_masker.inverse_transform(components_ica_zscored)

#for comp in range(n_components): # If you want to look through all 20!
for comp in [0,1]:
    fig = plt.figure(figsize=(20,10))
    display = plotting.plot_stat_map(image.index_img(ica_image, comp), figure = fig , threshold=0, cut_coords=[1,-23,-20], black_bg=0)
    
    display.title(f'ICA Component {comp+1}',size=30)
    display.annotate(size=30)
    cbar_ax = fig.axes[-1]
    cbar_ax.tick_params(labelsize=30)

The first spatial mode again looks like motion artifact, while the second looks like fusiform activity, which we would expect to have high signal in these data, owing to the blocked task (later we will remove this task-related activation to simulate resting-state by adjusting for confounds from a design matrix, though an alternative would be to project out any spatial modes from ICA that correlate with the task regressors, like this one). As noted above, there are a range of tools that attempt to separate signal from noise components. 

If we look at their corresponding temporal modes:

In [ ]:
#for comp in range(n_components):
for comp in range(2):
    plt.plot(ica.components_[comp,:]);
    #plt.plot(ica.components_[comp,1:200]); # If want to zoom-in to see task blocks

You can see that ICA has avoided the session effects that dominated the PCA. (If you zoom into just one run, eg first ~200 TRs, you can see that the second component does show the blocked nature of the task.)

There are much better ways to perform ICA across runs and subjects than shown here (e.g, you can clean runs before running ICA, or treat runs/subjects as a separate dimension in a tensor ICA, or use back-projection of concatenated data, etc). One way (which is normally used across subjects, but we will use across runs here) is "canonical ICA" ([Varoquaux et al., 2010](https://doi.org/10.1016/j.neuroimage.2010.02.010)) which uses Canonical Correlation Analysis (CCA), a correlation measure between two sets of variables, to ensure consistency across participants (runs) and find a lower dimensional space common to the participants (runs), before performing ICA on the common space to extract ICA components:

In [ ]:
canica = CanICA(
    n_components=n_components,
    memory="nilearn_cache",
    memory_level=2,
    verbose=0,
    mask_strategy="epi",
    random_state=0,
    standardize="zscore_sample",
)
canica.fit(all_runs)

warnings.filterwarnings("ignore", category=UserWarning) # just to avoid non-important warnings

canica_components_img = canica.components_img_
# components_img is a Nifti Image object, and can be saved to a file with:
# canica_components_img.to_filename("canica_resting_state.nii.gz")

display = plot_prob_atlas(canica_components_img, title="All ICA components", colorbar=True, vmin=1, vmax=21);
display._cbar.set_ticks(range(1,21))

Each colour corresponds to a Canonical ICA network (of voxels showing similar timecourses). 

This concludes the multivariate decomposition introduction (though we have only scratched the surface), and we now move onto ROI-based, pairwise connectivity...

## 2. ROI-based connectivity

### 2.1 Data cleaning

For ROI-based (or voxel-based) pairwise connectivity, we first summarise the timeseries within ROIs by averaging over voxels contained by that ROI (though one could also take the first singular vector of the non-mean-corrected voxel timeseries, which would allow for some heterogeniety within an ROI). There are several ways to define ROIs, as illustrated below.

We then have to clean the timeseries, i.e, remove noise that could cause spurious correlations between timeseries. There are three main types of non-neural noise in fMRI timeseries: 1) instrumental noise (like scanner drift), 2) motion-related, and 3) physiological artifacts (like breathing). We address each of these in turn, but let's start with looking at the raw data. For the following demonstrations, we will stick to one run (the last one), for simplicity.

In [ ]:
last_img = nilearn.image.load_img(all_runs[-1]) # choose last run where may be most motion for illustration
epi_mask = nilearn.masking.compute_epi_mask(last_img, exclude_zeros=True)

epi_masker = NiftiMasker(mask_img = epi_mask, standardize=None, detrend=False) # Raw, raw data
timeseries = epi_masker.fit_transform(last_img)

fig = plt.figure(figsize=(20,10)) # plot two random voxels
plt.plot(timeseries[:,100],    lw=2, ls='--', color='blue',  label='One Voxel')
plt.plot(timeseries[:,10000],  lw=2, ls='-',  color='red',   label='Another Voxel')
ax = plt.gca(); ax.tick_params(axis='x', labelsize=25);ax.tick_params(axis='y', labelsize=25); plt.title("Raw BOLD timeseries")
ax.legend(prop={'size':25}, fontsize='xx-large', frameon=False);

pcor = pearsonr(timeseries[:,100], timeseries[:,10000]).statistic
print(f"Pearson correlation between two random voxels {pcor:.3f}")

You can see that the (randomly chosen) brain voxels have quite different mean values, given that the fMRI BOLD signal has arbitrary units, and show slow drifts over time.

We can also show all voxels in a "carpet plot", which shows time (TR) horizontally and voxel vertically. First though, we are going to use an atlas to classify each voxel as gray-matter (GM), white-matter (WM) and Cerebospinal Fluid (CSF). For this, we're going to use a probabilistic atlas, in which a voxel can belong to several components. These atlases are represented by 4D images where the 3D components, also called ‘spatial maps’, are stacked along the 4th dimension:

In [ ]:
atlas_dir = os.path.join(wd, 'atlases')
if not os.path.exists(atlas_dir):
    os.makedirs(atlas_dir)

# Download an atlas definition of GM, WM and CSF voxels 
atlas = datasets.fetch_icbm152_2009(data_dir = atlas_dir) # this downloads an atlas 

# If want load an atlas already downloaed to a local path
#atlas_dir = '/imaging/correia/da05/workshops/2026-COGNESTIC/06_fMRI/atlases/icbm152_2009/mni_icbm152_nlin_sym_09a'
#img1 = image.load_img(atlas_dir + "/*gm*.nii.gz")
#img2 = image.load_img(atlas_dir + "/*wm*.nii.gz")
#img3 = image.load_img(atlas_dir + "/*csf*.nii.gz")
#atlas_img = image.concat_imgs([img1, img2, img3])

# We need to resample atlas to match EPI data, so use the first volume of the functional image as a 3D reference
ref_img = image.index_img(last_img, 0)
gm_img  = image.resample_to_img(atlas["gm"], ref_img, interpolation="nearest")
wm_img  = image.resample_to_img(atlas["wm"], ref_img, interpolation="nearest")
csf_img = image.resample_to_img(atlas["csf"], ref_img, interpolation="nearest")

# Rebuild an atlas image with the resampled GM, WM and CSF images
atlas_img = image.concat_imgs((gm_img, wm_img, csf_img))
map_labels = {"GM": 1, "WM": 2, "CSF": 3} # assign them labels

# These images contain probabilities of each tissue-type, which we can binarize by taking the maximum across tissue-types for each voxel
atlas_data = atlas_img.get_fdata() 
discrete_version = np.argmax(atlas_data, axis = 3) + 1    # index of maximum value across GM, WM and CSF for each voxel
discrete_version[np.max(atlas_data, axis = 3) == 0] = 0   # reset everything

# Remove voxels outside the EPI mask
epi_mask_data = epi_mask.get_fdata().astype(bool)
discrete_version = np.where(epi_mask_data, discrete_version, 0)
# Rebuild the atlas image
discrete_atlas_img = image.new_img_like(atlas_img, discrete_version)

# Plot image of segments (suppressing warning about data-type)
plotting.plot_stat_map(discrete_atlas_img, cmap='tab20b', colorbar=False);

This labels every voxel as GM (green), WM (red) and CSF (pink).
#### 2.1.1 Raw Carpet
We can now plot a carpet, with voxels grouped by tissue-type:

In [ ]:
epi_masked_images = epi_masker.inverse_transform(timeseries)

fig, ax = plt.subplots(figsize=(12, 8))
display = plotting.plot_carpet(
    img = epi_masked_images,
    mask_img = discrete_atlas_img,
    t_r = TR,
    mask_labels = map_labels,
    axes = ax, title='Raw BOLD timeseries',
    standardize = None,
    detrend = False,
    cmap="gray")

#... though note plot_carpet can do same transformations on the fly, eg with standardize set to "zscore_sample")
fig.show();

This is quite a mess! The data are on such different scales across voxels, that you cannot see any variation. However, let's see the effect of detrending, standardising and applying a high-pass filter:

#### 2.1.2 Filtered carpet

One major source of noise relates to instrumental noise like scanner drift (the BOLD signal typically drifts up and down over time, eg due to changes in temperature). This tends to be low-frequency, so we can remove by high-pass filtering the data. 

In fact, high-pass filtering also removes other types of physiological noise that are of higher frequency than our sampling rate. For example, pulse is typically 1Hz, which is above the Nyquist limit if we sample with a TR=2 (ie 0.5 Hz). This type of noise therefore becomes aliased into lower-frequencies, so also removed by high-pass filtering.

Some people also apply a low-pass filter (i.e, effectively band-pass filter), but it is unclear whether this is important (see [Geerligs et al (2017)](https://pmc.ncbi.nlm.nih.gov/articles/PMC5518296/) for example). 

Let's look at the effect of filtering out low-frequency noise, and scaling all voxels to their SD over time (Z-scoring/standardising):

In [ ]:
high_pass_cut   = 0.008; # cut-off (typical choice - filtering too much removes dfs in our data)
epi_masker = NiftiMasker(mask_img = epi_mask, standardize='zscore_sample', high_pass = high_pass_cut, t_r = TR) 
timeseries = epi_masker.fit_transform(last_img)

fig = plt.figure(figsize=(20,10))
plt.plot(timeseries[:,100],    lw=2, ls='--', color='blue',  label='One Voxel')
plt.plot(timeseries[:,10000],  lw=2, ls='-',  color='red',   label='Another Voxel')
ax = plt.gca(); ax.tick_params(axis='x', labelsize=25); 
ax.tick_params(axis='y', labelsize=25); plt.title("Filtered BOLD timeseries", fontsize=20)

pcor = pearsonr(timeseries[:,100], timeseries[:,10000]).statistic
print(f"Pearson correlation between two random voxels {pcor:.3f}")

You can see that the two random voxels are now better matched in terms of range, etc. Their (absolute) correlation has gone down, but this is not necessarily a bad thing, since some of the correlation could be caused by artifacts.

Or for the whole brain:

In [ ]:
masked_filtered_images = epi_masker.inverse_transform(timeseries) 
ig, ax = plt.subplots(figsize=(12, 8))
display = plotting.plot_carpet(img = masked_filtered_images, mask_img = discrete_atlas_img, t_r = TR, 
                               mask_labels = map_labels, axes = ax, title='Filtered BOLD timeseries',
    standardize = None, detrend = False, cmap="gray") # data already standardised (and highpass-filter detrends)
fig.show();

Standardisation and high-pass filtering have made the carpet plot more homogeneous (and useful!). If you focus on the GM voxels, you can see this is not a particularly bad run (in other cases you might see vertical stripes indicating large motion at a certain time, or high variance in some voxels). There seems to be a bit more variance at start of run, and possibly some artifact around 65 TRs. You might also be able to see different smoothness (temporal autocorrelation) for CSF vs WM vs GM voxels.

Note that filtering has also added end-effects towards end of run, but these do not matter because they same is true of all voxels that we will correlate later. However, there are still potential artifacts in the data, such as residual effects of motion (that cannot be corrected by realignment during preprocessing) and other slower biorhythms that affect BOLD signal (like breathing) that do not directly reflect neural activity.

#### 2.4.3 Deconfounded carpet

As in the notebook on subject-level models, we will collect the confounds created by fMRIPrep, but this time we will use some more than just the motion paramerers, since we want to ensure that any correlation between voxel timeseries (TSC) does not owe to noise sources like bioryhthms (eg breathing) that are shared across many voxels (we want it to reflect primarily neural activity). Note that these additional confounds are less important for the task-based analysis we did in previous notebooks, because they are unlikely to be phase-locked to the regressors in our design matrix, and so just appear in the residual error, whereas for TSC we have no design matrix and the Pearson correlation estimate of functional connectivity is highly sensitive since it is based on phase-locking across voxels.)

First we will load the 6 motion parameters, but this time take various expansions of them, like their squares and the difference between successive TRs. These are to capture nonlinear and time-lagged artifacts caused by motion (like EPI "spin-history effects" for example). These are sometimes called the "Sattherthwaite 24" set. They are already created by fMRIPrep, but we recreate them below for clarity. 

We will also add signal recorded from the white-matter (WM) and cerebrospinal fluid (CSF) partitions created by fMRIPrep, to capture biorhythms that should not contain any neural activity (unless contaminated by partial grey-matter volume): 

In [ ]:
confound_files = layout.get(subject = sID, datatype = 'func', desc = 'confounds', extension = ".tsv", return_type = 'filename')
motion_confounds = ['trans_x', 'trans_y', 'trans_z', 'rot_x', 'rot_y', 'rot_z']
other_confounds  = ['csf', 'white_matter']

confounds_per_run = []
#for conf_file in confound_files: # If want all runs
for conf_file in [confound_files[-1]]: # Just use last run below
    this_conf = pandas.read_table(conf_file)
    motion_subset = this_conf[motion_confounds].fillna(0) # replace NaN with 0
    motion_change = motion_subset.diff().fillna(0)
    motion_change.columns = 'diff_' + motion_subset.columns
    motion_square = motion_subset.pow(2)
    motion_square.columns = 'sq_' + motion_subset.columns 
    motion_change_square = motion_change.pow(2)
    motion_change_square.columns = 'sq_' + motion_change.columns
    motion_24 = pandas.concat([motion_subset, motion_change, motion_square, motion_change_square], axis=1)
    confounds = pandas.concat([motion_24, this_conf[other_confounds].fillna(0)], axis=1)
    confounds = (confounds - confounds.mean())/(confounds.std()) # Z-score just for visualisation
    confounds_per_run.append(confounds)
    
print(f"Using around {confounds_per_run[0].shape[1]} dfs per run to remove motion, CSF and WM effects")
confounds_per_run[-1].head() # show last run

Let's see what happens if we regress these confounds out of the data...

In [ ]:
warnings.filterwarnings("ignore", category=DeprecationWarning) 

confounds_to_remove = np.asarray(confounds_per_run[-1]) # take confounds from last run to match data
timeseries = epi_masker.fit_transform(last_img, confounds = confounds_to_remove) 

fig = plt.figure(figsize=(20,10))
plt.plot(timeseries[:,100],    lw=2, ls='--', color='blue',  label='One Voxel')
plt.plot(timeseries[:,10000],  lw=2, ls='-',  color='red',   label='Another Voxel')
ax = plt.gca(); ax.tick_params(axis='x', labelsize=25);
ax.tick_params(axis='y', labelsize=25); plt.title("Filtered, deconfounded BOLD timeseries", fontsize = 20)
ax.legend(prop={'size':25}, fontsize='xx-large', frameon=False);

pcor = pearsonr(timeseries[:,100], timeseries[:,10000]).statistic
print(f"Pearson correlation between two random voxels {pcor:.3f}")

epi_masked_images = epi_masker.inverse_transform(timeseries) 
ig, ax = plt.subplots(figsize=(12, 8))
display = plotting.plot_carpet(img = epi_masked_images, mask_img = discrete_atlas_img, t_r = TR, 
                               mask_labels = map_labels, axes = ax, title='Filtered, deconfounded BOLD timeseries',
    standardize = None, detrend = False, cmap="gray") # data already standardised (and highpass-filter detrends)

This looks a bit cleaner (e.g. at start of run), though effects may be more dramatic in other datasets, eg from children (try your data!)

This would be sufficient for resting-state data, but in our data, we still have responses related to the task, ie each trial that a stimulus was presented. 

#### 2.4.4 Removing task-effects

In our task data, if we correlated voxel time-series from the deconfounded data above, there might be high correlation between some brain regions simply because they independently responded to the same stimuli. To remove this, we can use the regressors we created in our GLM in previous notebooks.

One way to remove this variance would be to use the LSA model in the task-based connectivity notebook, i.e. adjust for a separate regressor for each trial. This would allow for fact that the amplitude of BOLD response might vary across trials of the same type (e.g, to fluctuations in attention), which the standard, one-regressor-per-condition (LSU) model would not allow for. However, the flexibility of LSA model can also fit noise (particularly with short-SOA event-related designs like ours) - at least fluctuations in the BOLD response that are not due to the trials themselves - ie remove true functional connectivity. We therefore use the standard LSU model here (but you can try the LSA model by uncommenting lines below, in which case you should observe that a strong resting-block structure emerges, suggesting it may indeed be over-fitting).

Note that in general, it is difficult to remove all possible task-related signal, because in addition to amplitude variation across trials, the HRF used may not be a perfect match for a particular subject and brain region. This could be resolved by using a more flexible basis set to model the BOLD impulse response, though combining both LSA and a flexible basis set is infeasible except with long SOAs (see Efficiency notebook).

In [ ]:
events_files = layout.get(subject=sID, datatype='func', suffix='events', extension=".tsv", return_type='filename')
print("Found " + str(len(events_files)) + " event files")
events_df = pandas.read_table(events_files[-1])
events = events_df.drop(columns = ['button_pushed', 'stim_file', 'trigger', 'circle_duration', 'response_time'])

slice_timing = layout.get_metadata(all_runs[-1])
if slice_timing['SliceTimingCorrected']:
  slice_time_ref = slice_timing['StartTime'] / TR

# If want to create a LSA model with all trials of ALL conditions
#for j, event in enumerate(events_df['trial_type']):
#   events_df.loc[j, 'trial_type'] = event + events_df['stim_file'][j][-8:-4]

# Create a design matrix without actually fitting data, which just requires from dummy frame times
nvols = last_img.shape[-1]
frame_times = np.linspace(0, (nvols - 1) * TR, nvols)
design_matrix = nilearn.glm.first_level.make_first_level_design_matrix(
  frame_times, events = events, hrf_model = 'glover', drift_model = 'cosine', high_pass = 0, drift_order = None)
# Note do not need highpass filter terms
nilearn.plotting.plot_design_matrix(design_matrix, output_file = None)
fig = plt.gcf(); fig.set_size_inches(8,4); plt.show();

We can now treat these task-related regressors as confounds too:

In [ ]:
trial_regressors = np.asarray(design_matrix)
trial_regressors = trial_regressors[:,:-1] # do not need constant

print(f"Using {trial_regressors.shape[1]} trial regressors, ie a total of {trial_regressors.shape[1] + confounds_to_remove.shape[1]} confound regressors from {nvols} TRs (though also df loss from highpass filtering)")

all_confounds = np.concatenate([confounds_to_remove, trial_regressors], axis=1)                 
timeseries    = epi_masker.fit_transform(last_img, confounds = all_confounds)

fig = plt.figure(figsize=(20,10))
plt.plot(timeseries[:,100],    lw=2, ls='--', color='blue',  label='One Voxel')
plt.plot(timeseries[:,10000],  lw=2, ls='-',  color='red',   label='Another Voxel')
ax = plt.gca(); ax.tick_params(axis='x', labelsize=25);ax.tick_params(axis='y', labelsize=25), 
plt.title('Filtered, deconfounded, task-adjusted BOLD timeseries', fontsize=20)
ax.legend(prop={'size':25}, fontsize=15, frameon=False);

pcor = pearsonr(timeseries[:,100], timeseries[:,10000]).statistic
print(f"Pearson correlation between two random voxels {pcor:.3f}")

epi_masked_images = epi_masker.inverse_transform(timeseries) 
fig, ax = plt.subplots(figsize=(12, 8))
display = plotting.plot_carpet(epi_masked_images, discrete_atlas_img, t_r = TR, standardize = 'zscore_sample', mask_labels = map_labels, axes = ax, 
                               title='Filtered, deconfounded, task-adjusted BOLD timeseries', cmap="gray")
fig.show();

Surprisingly, any effects of removing task-evoked activation, e.g, any vertical structure related to the blocking of trials, are not obvious to the eye (which shows how subtle task-related fMRI changes can be!). But at least we know they should have been removed (with caveats noted above).

### 2.2 ROI-to-Whole-Brain (seed-based) connectivity

We can now estimate the functional connectivity between a selected "seed" ROI and all (grey-matter) voxels; a kind of "mapping" approach. Here we are going to use the right Fusiform ROI that we defined in previous notebooks as responding to faces.

First we extract cleaned timeseries for every voxel, now also adding some spatial smoothing (to reduce noise, on assumption that functional connectivity operates over this spatial scale). We also extract a timeseries for the ROI, averaged over voxels.

In [ ]:
# smoothing helps, so re-mask with smoothing
#brain_masker = NiftiMasker(standardize='zscore_sample', high_pass = high_pass_cut, t_r = TR, smoothing_fwhm = 8) 
epi_masker = NiftiMasker(mask_img = epi_mask, standardize='zscore_sample', high_pass = high_pass_cut, t_r = TR, smoothing_fwhm = 8) 
timeseries  = epi_masker.fit_transform(last_img, confounds = all_confounds)

#fusi_ROI = nibabel.load('/imaging/correia/da05/workshops/2026-COGNESTIC/06_fMRI/FaceRecognition/results/FFA_sphere_and_faces-scrambled_fwe.nii.gz')
fusi_ROI = nibabel.load(os.path.join(wd, 'task_con', 'FFA_sphere_and_faces-scrambled_fwe.nii.gz'))

fusi_masker = NiftiMasker(fusi_ROI, standardize='zscore_sample', high_pass = high_pass_cut, t_r = TR) # no need for smoothing since will average over ROI
seed_data   = fusi_masker.fit_transform(last_img, confounds = all_confounds)
seed_data   = np.mean(seed_data, axis=1) # average across voxels
seed_data   = (seed_data - np.mean(seed_data)) / np.std(seed_data) # zscore

To calculate connectivity, we use Pearson's correlation coefficient, which can be implemented in a computationally simple way by the dot-product of normalised (Z-scored) vectors (see also task-based connectivity notebook)

In [ ]:
seed_to_voxel_correlations = np.dot(seed_data.T, timeseries)/seed_data.shape[0]
print("Correlation min = " + str(np.min(seed_to_voxel_correlations)) + ", max = " + str(np.max(seed_to_voxel_correlations)))
seed_to_voxel_correlations_fisher_z = np.arctanh(seed_to_voxel_correlations)

We can now produce an image of voxels whose values represent the strength of connection with the seed ROI (using the Fisher transform to convert the Pearson correlation coefficients to an unbounded range):

In [ ]:
# convert the np array back into a nifti image
seed_to_voxel_correlations_fisher_z_img = epi_masker.inverse_transform(seed_to_voxel_correlations_fisher_z)
# apply brain mask
#seed_to_voxel_correlations_fisher_z_img = nilearn.image.math_img('(in_mask * data)', 
    #in_mask=epi_mask, data=seed_to_voxel_correlations_fisher_z_img)
seed_to_voxel_fname = 'fcon_fusi_seed.nii.gz'
seed_to_voxel_correlations_fisher_z_img.to_filename(seed_to_voxel_fname) 

MNI_coord = (41.5,	-48.5,	-18.5) # From Notebook 06, second peak of Faces>Scrambled contrast
display = nilearn.plotting.plot_stat_map(seed_to_voxel_fname, cut_coords = MNI_coord, threshold = 0.5,
    title="Z-values for functional correlation with fusiform peak voxel")

Of course, voxels in the right fusiform will have very high values because they are spatially part of the seed region. However, the left fusiform is also correlated (even if would not survive correction), and this cannot be due to spatial overlap with the seed (nor should it simpy reflect common activation to face stimuli, assuming our regressors were sufficient to remove such task-related activation; see earlier discussion). 

### 2.3 Atlas-based ROI-to-ROI connectome

We can also divide (parcellate) the brain into a number of ROIs based on an atlas in MNI space. There are a large number of such atlases, some in which the ROIs are defined anatomically, others in which they are defined functionally, e.g. by clustering functional connectivity results in other datasets.

Note that there is an ongoing debate on the best way to define ROIs (i.e, how to parcellate the brain). For a recent review, see [Neuroparc](https://github.com/neurodata/neuroparc) and [Lawrence et al. (2021)](https://doi.org/10.1038/s41597-021-00849-3). There is also the question of whether functional parcellations vary between people, e.g., as a function of age, [Geerligs et al. (2015)](https://doi.org/10.1523/JNEUROSCI.1324-15.2015); see also [Salehi et al. 2020](https://doi.org/10.1016/j.neuroimage.2019.116366) and [Bohland et al. 2009](https://doi.org/10.1371/journal.pone.0007200). You can also create your own anatomical or functional atlases see [Moghimi et al. (2021)](https://arxiv.org/ftp/arxiv/papers/2107/2107.03475.pdf); for a recent review, see also [Passingham et al. (2002)](https://doi.org/10.1038/nrn893).

The optimal number of ROIs is also important, with a few hundred often being needed for connectomics [Craddock et al. (2021)](https://doi.org/10.1002/hbm.21333). However, we will start with a relatively small, probabilistic atlas called "MSDL" (https://nilearn.github.io/dev/modules/generated/nilearn.datasets.fetch_atlas_msdl.html):

In [ ]:
atlas = datasets.fetch_atlas_msdl(data_dir=atlas_dir)
atlas_filename = atlas["maps"]
plotting.plot_prob_atlas(atlas_filename, title='MSDL Atlas');

We can examine the ROI labels and centroids:

In [ ]:
atlas_img = image.load_img(atlas_filename)
print(f'Shape of probabilistic atlas {atlas_img.shape}')
labels = atlas["labels"]

temp = pandas.DataFrame(atlas)
print(temp["labels"].head(11))
print(temp["region_coords"].head(11))
#print(temp["networks"].head(11))

Now we can produce a connectivity matrix between all pairs of ROIs (a type of "connectome"). Let's use the cleaned data from earlier, but now a NiftiMapsMasker, plus the Connectivity Measure toolbox:

In [ ]:
warnings.filterwarnings("ignore", category=FutureWarning) 

atlas_masker = NiftiMapsMasker(maps_img = atlas_filename, 
                             standardize='zscore_sample', high_pass = high_pass_cut, t_r = TR) 
timeseries  = atlas_masker.fit_transform(last_img, confounds = all_confounds)

correlation_measure = ConnectivityMeasure(kind="correlation")
correlation_matrix = correlation_measure.fit_transform(timeseries)[0]
np.fill_diagonal(correlation_matrix, 0)

fig, ax = plt.subplots(figsize=(12, 8))
# Plot correlation matrix - note: matrix is ordered for block-like representation
display = plotting.plot_matrix(correlation_matrix, labels=labels,
                     vmax=0.8, vmin=-0.8, reorder=False,axes=ax);
ax.set_xticklabels(ax.get_xticklabels(),fontsize=12);
ax.set_yticklabels(ax.get_yticklabels(),fontsize=12);
ax.set_title('Counfounds regressed',fontsize=15);

You can see most connections are positive (based on Pearson's correlation coefficient), and that there is some clustering, but this depends on the ordering of ROIs.

#### 2.3.1 Global Signal

Another potential signature of physiological noise is the "global signal", the average timeseries over all brain voxels. Such "global signal regression" is contentious however; see [Murphy et al. 2017](https://doi.org/10.1016/j.neuroimage.2016.11.052): while it can be quite effective at removing noise, it can also remove true neural signal that happens to occur in many voxels. It also produces many negative Pearson correlation coefficients, since some pairs of voxels will be correlated, but less than the correlation between any voxel and the average over voxels. Negative correlations might be interpretable in terms of less-than-average connectivity (i.e, not necessarily true neural anti-correlation), but some argue that we should focus on positive correlations only (eg for graph-theoretic analysis in next notebook).

Again, global signal is in the confounds file fromy fMRIprep, so let's see what happens when we regress it out too:

In [ ]:
confounds_global = pandas.DataFrame(all_confounds)
global_signal = pandas.read_table(confound_files[-1], usecols = ["global_signal"])
global_signal = (global_signal -  global_signal.mean()) /  global_signal.std()
confounds_global['Global'] = global_signal

timeseries_global  = atlas_masker.fit_transform(last_img, confounds = confounds_global)
correlation_matrix_global = correlation_measure.fit_transform(timeseries_global)[0]
np.fill_diagonal(correlation_matrix_global, 0)

fig, ax = plt.subplots(figsize=(12, 8))
# Plot correlation matrix - note: matrix is ordered for block-like representation
display= plotting.plot_matrix(correlation_matrix_global, labels=labels,
                     vmax=0.8, vmin=-0.8, reorder=False,axes=ax);
ax.set_xticklabels(ax.get_xticklabels(),fontsize=12);
ax.set_yticklabels(ax.get_yticklabels(),fontsize=12);
ax.set_title('Counfounds + Global regressed',fontsize=15);

Now you can see a lot more negative connections, i.e., correlations that are less than the average. We can also plot these matrices in terms of lines between the centroids of each ROI (weighted by their strength): 

In [ ]:
coords = atlas.region_coords

# We can threshold connections in terms of proportions (eg top 20%) or absolute value (of correlation coefficient)
#plotting.plot_connectome(correlation_matrix, coords, edge_threshold="80%", colorbar=True, title='Confound Regression')
plotting.plot_connectome(correlation_matrix, coords, edge_threshold=0.4, colorbar=True, title='Confound Regression')
plotting.show()

#plotting.plot_connectome(correlation_matrix_global, coords, edge_threshold="80%", colorbar=True, title='Confound + Global Regression')
plotting.plot_connectome(correlation_matrix_global, coords, edge_threshold=0.4, colorbar=True, title='Confound + Global Regression')
plotting.show()

The strong connections between homologous regions across hemispheres are noticeable, which tend to be stronger than within-hemisphere connections (so are a good check that your connectome makes sense). There is also an interactive viewer if you want to explore further:

In [ ]:
view = plotting.view_connectome(
    correlation_matrix_global, coords, edge_threshold="80%" # Now a relative threshold for comparison
)
# In a Jupyter notebook, if ``view`` is the output of a cell, it will be displayed below the cell
view

#### 2.3.2 Partial Correlation

You can also estimate the correlation between two ROIs after adjusting for the timeseries in all other ROIs. This is an attempt to measure direct connections, adjusting for indirect paths between ROIs. There are not always sufficient df's to do this with a GLM (when number of connections is higher than number of timepoints) but there are regularised estimators than can do this: 

In [ ]:
connectivity_measure_partial = ConnectivityMeasure(kind='partial correlation')
correlation_matrix_partial = connectivity_measure_partial.fit_transform(timeseries)[0] # using timeseries without global adjustment
np.fill_diagonal(correlation_matrix_partial,np.nan)
fig, ax = plt.subplots(figsize=(12, 8))

# Plot correlation matrix - note: matrix is ordered for block-like representation
display= plotting.plot_matrix(correlation_matrix_partial, labels=labels,
                     vmax=0.8, vmin=-0.8, reorder=False,axes=ax);
ax.set_xticklabels(ax.get_xticklabels(),fontsize=12);
ax.set_yticklabels(ax.get_yticklabels(),fontsize=12);
ax.set_title('Partial Correlation (and Confounds)',fontsize=15);

#plotting.plot_connectome(correlation_matrix_partial, coords, edge_threshold="80%", colorbar=True, title='Partial Correlation (and Confounds)')
plotting.plot_connectome(correlation_matrix_partial, coords, edge_threshold=0.4, colorbar=True, title='Partial Correlation (and Confounds)')
plotting.show()

This matrix is much sparser, and fewer connections surpass an absolute threshold. 

Note that you could also threshold on a statistic, eg p-value, though in this example, the df's are difficult to estimate since they depend not only on the number of confounds removed, but also the filtering employed and the intrinsic autocorrelation in the data. In other situations, you might be testing connection strengths across independent subjects, for which it is easier to define a p-value (normally after applying the Fisher transform), and then various methods for correcting for multiple comparisons can be applied (see Stats notebook; and also [Zalesky et al., 2010](https://doi.org/10.1016/j.neuroimage.2010.06.041) for network-based statistics).

#### 2.3.3 Larger Connectomes

We now switch to the larger "Schaefer" atlas of 500 ROIs, which is bundled with nilearn, and comes with a labelling in which those ROIs have already been defined according to 17 "Yeo" networks (which we need to extract from the ROI names; a little hacky!): 

In [ ]:
# download the 500 ROI version of Schaefer atlas, which can be organised into 17 networks according to Yeo
schaefer_atlas = datasets.fetch_atlas_schaefer_2018(n_rois = 500, yeo_networks = 17, data_dir = atlas_dir, resolution_mm=2)

# If want to see all ROIs:
#plotting.plot_roi(schaefer_atlas_filename, colorbar = True, interpolation = 'nearest');
atlas_img = image.load_img(schaefer_atlas.maps)

# 1. Extract network labels 
labels = [
    l.decode("utf-8") if isinstance(l, bytes) else l for l in schaefer_atlas.labels
]

# 2. Identify the unique 17 network names from the labels
# Schaefer strings look like: '17Networks_LH_VisCent_ExStr_1'
network_names = []
for label in labels:
    # Extract the network name (e.g., 'VisCent', 'DefaultA')
    parts = label.split("_")
    net_name = parts[2] if len(parts) > 2 else "Background"
    network_names.append(net_name)

# Get a unique, sorted list of the 17 network categories
unique_networks = sorted(list(set(network_names)))

# 3. Create a lookup table (LUT) array to map 500 ROIs to 17 networks
lut_size = len(labels)
roi_to_network_lut = np.zeros(lut_size, dtype=int)

for roi_id, net_name in enumerate(network_names):
    network_id = unique_networks.index(net_name)  # 1-indexed network ID
    roi_to_network_lut[roi_id] = network_id

# 4. Remap the original 3D image data using our LUT
atlas_data = atlas_img.get_fdata().astype(int)
network_coded_data = roi_to_network_lut[atlas_data]

# Convert the recoded numpy array back into a Nifti image
network_img = image.new_img_like(atlas_img, network_coded_data)

#plotting.plot_roi(network_img, colorbar = True, interpolation = 'nearest');
plotting.plot_roi(
    roi_img=network_img,
    title="Schaefer 2018 mapped to 17 Networks",
    display_mode="mosaic",  # Shows a clean grid of slices
    cmap="tab20",  # Qualitative color map perfect for discrete categories
    colorbar=True,
);

Now we can calculate correlation between every pair of ROIs as before, and plot a connectome, with ROIs ordered as they are in original atlas, in which left ROIs are numbered first, then right ROIs. As a consequence, you can see a second "diagonal" stripe half-way across from the main diagonal, which again represents connectivity between homologous ROIs across hemispheres.

In [ ]:
correlation_measure = nilearn.connectome.ConnectivityMeasure(kind='correlation')

schaefer_masker = nilearn.maskers.NiftiLabelsMasker(labels_img = atlas_img, # schaefer_atlas_filename,
                            standardize = 'zscore_sample',
                            detrend = False,
                            high_pass = high_pass_cut,
                            t_r = TR, verbose = 0)
time_series_clean_schaefer = schaefer_masker.fit_transform(last_img, confounds = all_confounds)

correlation_matrix_schaefer = correlation_measure.fit_transform([time_series_clean_schaefer])[0]
np.fill_diagonal(correlation_matrix_schaefer, 0)

# Group correlation matrix by ROI number (showing laterality effects)
plt.figure(figsize=(10,8)); ax = plt.gca()
display = plotting.plot_matrix(correlation_matrix_schaefer,
                     vmax=0.8, vmin=-0.8, reorder=False, axes=ax, title='Ordered by left then right ROIs');
ax.set_yticklabels(ax.get_yticklabels(),fontsize=10);

We can also reorder the ROIs by networks (using the new numbered we calculated earlier). Now you should see block-like structures along the main diagonal, reflecting higher connectivity within networks than between networks.

In [ ]:
# Group ROI correlation matrix by Network
plt.figure(figsize=(10,8)); ax = plt.gca()
reorder_index_schaefer = np.argsort(roi_to_network_lut)[1:]-1
reorder_corr = correlation_matrix_schaefer[reorder_index_schaefer][:,reorder_index_schaefer];
display = plotting.plot_matrix(reorder_corr,
                     vmax=0.8, vmin=-0.8, reorder=False, axes=ax, title='Ordered by Network');
ax.set_yticklabels(ax.get_yticklabels(),fontsize=10);

Finally, we could plot connectivity between networks (rather than ROIs):

In [ ]:
roi_to_network = roi_to_network_lut[1:] # Drop background
unique_labels = np.unique(roi_to_network)
n_networks = len(unique_labels)

network_means = np.zeros((n_networks, n_networks))
for x, x_label in enumerate(unique_labels):
    x_indices = (roi_to_network == x_label)
    for y, y_label in enumerate(unique_labels):
        y_indices = (roi_to_network == y_label)
        sub_matrix = correlation_matrix_schaefer[np.ix_(x_indices, y_indices)]
        sub_matrix_mean = np.mean(sub_matrix[sub_matrix!=0]) # Ignore leading diagonal of within-network connectivity
        #network_means.append(sub_matrix_mean)
        network_means[x,y] = sub_matrix_mean

plt.figure(figsize=(10,8)); ax = plt.gca()
display = plotting.plot_matrix(network_means, labels=unique_networks[1:],
                     vmax=0.8, vmin=-0.8, reorder=False,axes=ax);
ax.set_xticklabels(ax.get_xticklabels(),fontsize=12);
ax.set_yticklabels(ax.get_yticklabels(),fontsize=12);
ax.set_title('Network Connectivity',fontsize=15);

This is the end of this notebook, but the networks notebook will show how to use matrices like these to estimate graph-theoretic properties.